# Vector Search RAG Demo Notebook

This notebook demonstrates vector search and retrieval-augmented generation using a simple in-memory vector store.

## 1. Preparation

### 1.1 Import Libraries

In [23]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

### 1.2 Define a small sentence embedding model

In [24]:
MODEL_NAME = 'all-MiniLM-L6-v2'
NUM_DOCUMENTS = 10
NUM_RESULTS = 3

def simple_llm(query, context_docs):
    print("\n--- LLM Generation (Placeholder) ---")
    print(f"Query: {query}")
    if context_docs:
        print("Context Documents Provided:")
        context_summary = "\n".join([f"- {doc}" for doc in context_docs])
        print(context_summary)
        # Simulate using context
        response = f"Based on the provided context about '{context_docs[0].split()[1]}...', the answer to '{query}' is likely related to that topic."
    else:
        print("No relevant context documents found.")
        response = f"I don't have specific context for '{query}', so I cannot provide a detailed answer based on retrieved documents."
    print(f"\nGenerated Response: {response}")
    return response

#### 1.2.1 Load Embedding Model

In [25]:

model = SentenceTransformer(MODEL_NAME)
print(f"Loaded model '{MODEL_NAME}'.")
DIMENSIONS = model.get_sentence_embedding_dimension()
print(f"Model embedding dimension: {DIMENSIONS}")

Loaded model 'all-MiniLM-L6-v2'.
Model embedding dimension: 384


### 1.3 Data Preparation

In [26]:
documents = [
    "The Eiffel Tower is located in Paris, France and is a famous landmark.",
    "Photosynthesis is the biological process plants use to convert light into energy.",
    "The Great Wall of China spans thousands of miles and was built over centuries.",
    "Artificial intelligence (AI) aims to create machines that mimic human cognitive functions.",
    "Tokyo, the capital of Japan, is a major global financial center.",
    "Global warming, a key aspect of climate change, involves rising average temperatures.",
    "The Amazon rainforest, located in South America, is vital for global biodiversity.",
    "Python is a versatile high-level programming language used in web development and data science.",
    "Leonardo da Vinci painted the Mona Lisa, which is displayed in the Louvre Museum.",
    "Quantum computing leverages quantum mechanics for computation, promising exponential speedups."
]
print(f"Using {len(documents)} sample documents.")

Using 10 sample documents.


## 2. Generating Embeddings

In [27]:
document_embeddings = model.encode(documents, show_progress_bar=True)
print(f"Generated {len(document_embeddings)} embeddings")
print(document_embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated 10 embeddings
[[ 6.3141562e-02  6.1381884e-02 -5.5807396e-03 ...  5.3957455e-02
   7.7411622e-02  3.2433987e-02]
 [-4.0092256e-02  5.3838253e-02 -8.2518853e-02 ...  4.0285494e-02
   1.3365789e-01  4.0841285e-02]
 [ 4.8844181e-02  9.0949759e-02  4.6097953e-03 ... -1.5060684e-02
   2.6803065e-02  5.0760478e-02]
 ...
 [-6.3321374e-02 -5.9485780e-03 -4.7749210e-02 ...  1.3398208e-01
   1.3424325e-01  2.2120697e-02]
 [-3.3226836e-02  2.5382752e-02  4.1847681e-03 ...  2.9930586e-02
   9.9914987e-03 -6.6730984e-02]
 [-8.9692332e-02  3.4088343e-02 -5.3773172e-02 ... -5.9082940e-02
   1.6216712e-02  6.6384149e-05]]


## 3. Setting up Simple Vector Store

In [28]:
# Store embeddings along with original text for easy retrieval
vector_database = {
    "ids": list(range(len(documents))),
    "embeddings": document_embeddings,
    "documents": documents
}
print(f"Created in-memory vector store with {len(vector_database['ids'])} entries.")

Created in-memory vector store with 10 entries.


## 4. Performing Vector Search

In [ ]:
query = "Where is the Eiffel Tower located?"
print(f"Query: '{query}'")

# Generate embedding for the query
query_embedding = model.encode([query])[0]  # Encode returns a list, take the first element
print(f"Generated query embedding.")

# Calculate similarities (cosine similarity)
similarities = cosine_similarity(
    query_embedding.reshape(1, -1),
    vector_database["embeddings"]
)[0]

# Create pairs of (doc_id, score)
similarity_scores = list(zip(vector_database["ids"], similarities))

# Sort by similarity
sorted_results = sorted(similarity_scores, key=lambda item: item[1], reverse=True)
print(f"Calculated similarities with {len(vector_database['ids'])} documents.")

# Get top N results
top_results = sorted_results[:NUM_RESULTS]
retrieved_doc_ids = [doc_id for doc_id, score in top_results]
retrieved_docs = [vector_database["documents"][doc_id] for doc_id in retrieved_doc_ids]

print(f"\nTop {NUM_RESULTS} most similar documents (using '{MODEL_NAME}'):")
for doc_id, score in top_results:
    print(f"- ID: {doc_id}, Score: {score:.4f}, Doc: {vector_database['documents'][doc_id]}")

Query: 'Where is the Eiffel Tower located?'
Generated query embedding.
Calculated similarities with 10 documents.

Top 3 most similar documents (using 'all-MiniLM-L6-v2'):
- ID: 0, Score: 0.8433, Doc: The Eiffel Tower is located in Paris, France and is a famous landmark.
- ID: 8, Score: 0.2439, Doc: Leonardo da Vinci painted the Mona Lisa, which is displayed in the Louvre Museum.
- ID: 2, Score: 0.2422, Doc: The Great Wall of China spans thousands of miles and was built over centuries.


## 5. Performing RAG

In [30]:
# Use the semantically relevant retrieved documents as context for the LLM
generated_response = simple_llm(query, retrieved_docs)


--- LLM Generation (Placeholder) ---
Query: Where is the Eiffel Tower located?
Context Documents Provided:
- The Eiffel Tower is located in Paris, France and is a famous landmark.
- Leonardo da Vinci painted the Mona Lisa, which is displayed in the Louvre Museum.
- The Great Wall of China spans thousands of miles and was built over centuries.

Generated Response: Based on the provided context about 'Eiffel...', the answer to 'Where is the Eiffel Tower located?' is likely related to that topic.
